In [ ]:
import sys
import os

# Use absolute paths to ensure imports work from anywhere
pipeline_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline"
eda_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline\eda"

sys.path.insert(0, eda_dir)
sys.path.insert(0, pipeline_dir)

import polars as pl
import numpy as np
import pandas as pd
from config import *
from eda_base import *
from eda_stats import *

# Mass Layoff Establishment Summary (S4)
Summary statistics of establishments with confirmed mass layoff events

In [ ]:
# Load mass layoff establishment summary
ml_path = os.path.join(OUT_DIR, 'establishment_ml_summary.parquet')
if os.path.exists(ml_path):
    ml_estab = pl.read_parquet(ml_path)
    print(f"Mass layoff establishment summary: {ml_estab.shape}")
    print(f"\nColumns: {ml_estab.columns}")
    print(f"\nFirst rows:")
    print(ml_estab.head())
else:
    print(f"File not found: {ml_path}")

In [ ]:
# Overview of mass layoff events
if 'ml_estab' in locals():
    n_ml_events = len(ml_estab)
    n_unique_sirens = ml_estab.select(pl.col('siret').n_unique()).to_series()[0]
    
    print(f"Mass layoff events (establishment-year pairs): {n_ml_events:,}")
    print(f"Unique establishments affected: {n_unique_sirens:,}")
    
    # Time distribution
    if 'an_layoff' in ml_estab.columns:
        events_by_year = ml_estab.group_by('an_layoff').agg(
            pl.len().alias('n_events')
        ).sort('an_layoff')
        
        print("\nMass layoff events by year:")
        print(events_by_year)

In [ ]:
# Employment rate at displacement (what % of workers stayed?)
if 'ml_estab' in locals() and 'prior_year_employment_rate' in ml_estab.columns:
    rate_stats = ml_estab.select([
        pl.col('prior_year_employment_rate').min().alias('min_rate'),
        pl.col('prior_year_employment_rate').max().alias('max_rate'),
        pl.col('prior_year_employment_rate').mean().alias('mean_rate'),
        pl.col('prior_year_employment_rate').median().alias('median_rate')
    ])
    
    print("Prior-year employment rate (worker retention rate):")
    for col in rate_stats.columns:
        val = rate_stats[col][0]
        pct = val * 100 if val is not None else None
        if pct is not None:
            print(f"  {col}: {pct:.1f}%")

In [ ]:
# Confirmation rate: what % of eligible workers are confirmed?
if 'ml_estab' in locals() and 'confirmation_rate' in ml_estab.columns:
    confirm_stats = ml_estab.select([
        pl.col('n_potential').sum().alias('total_potential'),
        pl.col('n_confirmed').sum().alias('total_confirmed')
    ])
    
    total_potential = confirm_stats['total_potential'][0]
    total_confirmed = confirm_stats['total_confirmed'][0]
    overall_rate = total_confirmed / total_potential if total_potential > 0 else 0
    
    print(f"Confirmation of mass layoff events:")
    print(f"  Eligible workers (n_potential): {total_potential:,}")
    print(f"  Confirmed displaced (n_confirmed): {total_confirmed:,}")
    print(f"  Overall confirmation rate: {overall_rate*100:.1f}%")

In [ ]:
# Distribution of establishment sizes
if 'ml_estab' in locals() and 'n_confirmed' in ml_estab.columns:
    # Number of workers displaced per event
    size_stats = ml_estab.select([
        pl.col('n_confirmed').min().alias('min_displaced'),
        pl.col('n_confirmed').max().alias('max_displaced'),
        pl.col('n_confirmed').mean().alias('mean_displaced'),
        pl.col('n_confirmed').median().alias('median_displaced'),
        pl.col('n_confirmed').sum().alias('total_displaced')
    ])
    
    print("Distribution of confirmed displaced workers per event:")
    print(f"  Min: {size_stats['min_displaced'][0]:.0f}")
    print(f"  Max: {size_stats['max_displaced'][0]:.0f}")
    print(f"  Mean: {size_stats['mean_displaced'][0]:.1f}")
    print(f"  Median: {size_stats['median_displaced'][0]:.0f}")
    print(f"  Total displaced: {size_stats['total_displaced'][0]:,.0f}")

In [ ]:
# Data quality: missing values
if 'ml_estab' in locals():
    print("Missing values (%):\n")
    for col in ml_estab.columns:
        null_pct = ml_estab.select(
            (pl.col(col).is_null().sum() / len(ml_estab) * 100).alias('pct')
        ).to_series()[0]
        if null_pct > 0:
            print(f"  {col}: {null_pct:.1f}%")

In [ ]:
# Summary: who is affected?
if 'ml_estab' in locals():
    n_events = len(ml_estab)
    n_estabs = ml_estab.select(pl.col('siret').n_unique()).to_series()[0]
    total_displaced = ml_estab.select(pl.col('n_confirmed').sum()).to_series()[0]
    
    print(f"\nMass Layoff Summary:")
    print(f"  Establishment-year pairs: {n_events:,}")
    print(f"  Unique establishments: {n_estabs:,}")
    print(f"  Total confirmed displaced: {total_displaced:,.0f}")
    print(f"  Average displaced per event: {total_displaced/n_events:.0f}")